# MeSH ontology scope

This experiment chooses the MeSH representation for subsequent DTP-KG experiments.

The supplied aggregate tables can be used to redraw both figures.

Once you have the appropriate licences and have prepared the data using the
[data guide](../../data/README.md) and
[DataPreparation.ipynb](../../data/DataPreparation.ipynb), you can generate the
MeSH embeddings and rerun the comparison using the functions in
[`src/dtpkg/mesh_scope/`](../../src/dtpkg/mesh_scope/).

See the [setup instructions](../../README.md) for dependencies and the
[experiment guide](README.md) for commands and saved-run provenance.

## Paths and run options

Replace each `...` in the next cell with your own quoted path, for example
`DATA_DIR = "/path/to/private/data"`. `PROJECT_ROOT` is the checkout folder;
`DATA_DIR` follows the [data guide](../../data/README.md).
Then adjust the derived input and output paths as needed and rerun the cells.
The path cell intentionally requires completion before execution; no config-file
or environment-variable changes are needed. The default run uses only the public results.


In [ ]:
from pathlib import Path
import sys

# Replace each ... with your own quoted path or Path(...).
PROJECT_ROOT = ...  # DTP-KG checkout folder
DATA_DIR = ...  # Local data folder; see data/README.md

# Check and normalize the paths supplied above.
if any(path is ... for path in (PROJECT_ROOT, DATA_DIR)):
    raise ValueError("Replace ... in PROJECT_ROOT, DATA_DIR with your local paths.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

# Import this checkout's source files.
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DATA_DIR = Path(DATA_DIR).expanduser().resolve()

# Public aggregate inputs and regenerated figures.
EXPERIMENT = PROJECT_ROOT / "experiments" / "03_mesh_scope"
TABLES = EXPERIMENT / "tables"
FIGURES = EXPERIMENT / "figures"
OUTPUT_DIR = FIGURES / "reproduced"

# Inputs and outputs for optional private preparation/training.
PRIVATE_RUN = EXPERIMENT / "results" / "mesh_scope"

REPLOT = False
RUN_EMBEDDINGS = False
RUN_TRAINING = False


In [2]:
import pandas as pd
from IPython.display import display, Markdown, Image

from dtpkg.mesh_scopes import SCOPES, SCOPE_BY_KEY
from dtpkg.mesh_scope.plot_selection import load_selection, plot_scope_selection
from dtpkg.mesh_scope.plot_heatmap import load_comparisons, render_scope_heatmap


## Candidate representations

Terms are MeSH tree positions. Each scope applies inclusive depth and frequency
filters, then independently fits TF-IDF and a 128-component SVD to binary
drug–term features. Because both the filters and embeddings differ, this
comparison does not isolate ontology depth alone.

In [3]:
scope_definitions = pd.DataFrame([
    {"Representation": s.name,
     "MeSH depths": f"{s.depth_min}–{s.depth_max}",
     "Drugs per tree position": f"{s.freq_min}–{s.freq_max}"}
    for s in SCOPES
])
display(scope_definitions.set_index("Representation"))

,MeSH depths,Drugs per tree position
Representation,,
Shallow,1–5,2–35
Intermediate,1–7,2–63
Full,1–10,2–52


## Matched training and validation

The three scopes share the common eligible drug population, evaluation pairs,
three repeated five-fold partitions, initialization, and positive-sampling
schedules. This gives 15 matched folds per scope and **45 fits**. The MeSH-only
MLP receives symmetric pair features `[a+b, abs(a-b)]`; it does not use the
constructed network.

Each outer development partition reserves 15% for validation. A training-only
scaler is fitted over the full sampling schedule and frozen. Training uses at
most 100 epochs with patience 10 and restores the checkpoint selected by
validation AUROC. The scope with the highest mean overall validation AUROC is
selected; exact ties favor the shallower scope. These scores describe selection,
not an unbiased estimate of the selected procedure's test performance.

In [4]:
selection = load_selection(TABLES)
fold_scores = selection["fold_values"]
summary = pd.DataFrame({
    "Mean validation AUROC": selection["means"],
    "SD across folds": fold_scores.std(ddof=1),
    "Matched folds": fold_scores.count(),
    "Selected": [key == selection["selected"] for key in selection["means"].index],
})
summary.index = [SCOPE_BY_KEY[key].name for key in summary.index]
summary.index.name = "Representation"
display(summary.round(4))
display(Markdown(f"**Selected representation: {SCOPE_BY_KEY[selection['selected']].name}.**"))

,Mean validation AUROC,SD across folds,Matched folds,Selected
Representation,,,,
Shallow,0.9656,0.0065,15,False
Intermediate,0.9776,0.0053,15,True
Full,0.9720,0.0052,15,False


**Selected representation: Intermediate.**

## Paper figure: validation scope selection

![Validation scope selection](figures/mesh_scope_validation_selection.png)

Each small point is an overall validation AUROC at a restored checkpoint; lines
connect matched folds. Diamonds show the mean. The saved study selects
Intermediate (0.9776), followed by Full (0.9720) and Shallow (0.9656).
These are overall pair-level AUROCs, not averages of category AUROCs.
[Full caption](figures/mesh_scope_validation_selection_caption.txt).

## Additional analysis: held-out category heatmap

The Low/Mid/Deep categories refer to each drug's deepest annotation: levels
1–5, 6–7, and 8–10. They differ from the representation scopes above.
Cells show the first named scope minus the second, averaged across matched
outer-test folds. Positive differences favor the first scope. F1 uses threshold
0.5. Checkpoints remain those selected on validation AUROC.

The heatmap displays AUROC and F1, while Holm correction includes the full
72-hypothesis family: six categories × three contrasts × four metrics
(AUROC, F1, AP, MCC). The following call recomputes and verifies that family
from aggregate fold scores. Test comparisons do not select the scope.

In [5]:
comparisons = load_comparisons(TABLES)
assert len(comparisons["comparisons"]) == 72
assert comparisons["comparisons"]["family_size"].eq(72).all()
print("Verified 72 paired contrasts, including AP and MCC in the Holm correction.")

Verified 72 paired contrasts, including AP and MCC in the Holm correction.


![MeSH scope test-category heatmap](figures/mesh_depth_effect_heatmap.png)

Asterisks indicate approximate corrected paired-CV tests after Holm adjustment
(* adjusted p < 0.05; ** adjusted p < 0.01). These are exploratory comparisons:
folds share drugs and training data, and the correction does not capture all
dependence. No asterisk does not establish equivalence. Embeddings are prepared
globally, making this a transductive representation comparison.
[Full caption](figures/mesh_depth_effect_heatmap_caption.txt).

## Optional: regenerate both figures

Set `REPLOT = True` in the paths cell and rerun the notebook.
This uses only `TABLES` and writes to `OUTPUT_DIR`, which defaults to the ignored
`figures/reproduced/` directory. The supplied reference figures remain unchanged.

In [6]:
if REPLOT:
    selection_png = plot_scope_selection(TABLES, OUTPUT_DIR)
    heatmap_png = render_scope_heatmap(TABLES, OUTPUT_DIR)
    display(Image(filename=str(selection_png)))
    display(Image(filename=str(heatmap_png)))

## Generate the MeSH embeddings

This step reads the annotations saved by
[data preparation](../../data/DataPreparation.ipynb) and creates the Shallow,
Intermediate, and Full TF-IDF/SVD representations with 128 features per drug.
The resulting CSVs are saved under `DATA_DIR / "mesh"` for use by later experiments.

Set `RUN_EMBEDDINGS = True` to generate them without training the models.
`RUN_TRAINING = True` also generates missing embeddings before training.
Existing embedding files are reused.


In [ ]:
if RUN_EMBEDDINGS or RUN_TRAINING:
    from dtpkg.mesh_scope.embeddings import build_embeddings

    missing_scopes = [scope for scope in SCOPES
                      if not (DATA_DIR / "mesh" / f"MeSH_{scope.level_key}_tfidf_svd128.csv").is_file()]
    if missing_scopes:
        build_embeddings(
            DATA_DIR / "mesh" / "extended_drug_info.csv",
            DATA_DIR / "mesh",
            scopes=missing_scopes,
            n_components=128,
        )


## Rerun the experiment with private inputs

In [7]:
if RUN_TRAINING:
    from dtpkg.mesh_scope.experiment import run_scope_comparison
    from dtpkg.mesh_scope.verify import verify

    run_scope_comparison(PRIVATE_RUN, data_dir=DATA_DIR)
    verify(PRIVATE_RUN, data_dir=DATA_DIR)
    plot_scope_selection(PRIVATE_RUN, OUTPUT_DIR / "new_run")
    render_scope_heatmap(PRIVATE_RUN, OUTPUT_DIR / "new_run")

## Reproduction limits and provenance

DrugBank data and derived record-level inputs and outputs are not distributed.
Full training requires appropriate DrugBank access and locally prepared inputs.
The public aggregate tables are sufficient to redraw both figures.

These tables preserve the completed study run; this packaging step did not
retrain it. The historical source archive is incomplete for `FusionTrainer.py`;
new runs record the packaged training sources and input hashes. See the
[saved-result provenance](README.md#saved-result-provenance) and
[snapshot metadata](tables/snapshot_provenance.json).